# Section 7 · End-to-End Agent Evaluation with AgentCore Evaluations

The complete evaluation lifecycle in one notebook:

1. **Deploy** a travel assistant agent to AgentCore Runtime
2. **Create** a custom LLM-as-judge evaluator
3. **Invoke** the agent with in-scope and out-of-scope prompts
4. **Evaluate** the session with built-in and custom evaluators
5. **Summarise** and save the results

### The agent
A travel assistant (`agent_app.py`) that answers questions about ✈️ flights, 🏨 hotels and 🌤️ weather — and should **refuse** anything else. The custom evaluator checks exactly that.

| | |
|:--|:--|
| Agent framework | Strands Agents |
| Model | Claude Haiku 4.5 on Amazon Bedrock |
| Deployment | AgentCore Runtime (CodeBuild — no local Docker) |
| Evaluation | On-demand, built-in + custom evaluators |

### Prerequisites
- AWS credentials; IAM permissions for `bedrock-agentcore:*`, `ecr:*`, `iam:CreateRole`/`PassRole`, `codebuild:*`, `s3:*`, `logs:*`, `xray:*`
- **CloudWatch Transaction Search** turned on (once per account and Region). Evaluations read the agent's traces from there. The launch step below tries to enable it automatically; if it can't, enable it in the CloudWatch console under *Application Signals → Transaction Search*.

---
## Step 0 — Install dependencies

In [ ]:
%pip install -q -r requirements.txt

---
## Step 1 — Imports and AWS session

In [ ]:
import json
import os
import time
import uuid

import boto3
from boto3.session import Session
from IPython.display import Markdown, display

from bedrock_agentcore_starter_toolkit import Evaluation, Runtime

boto_session = Session()
region = boto_session.region_name or "us-east-1"
account_id = boto3.client("sts", region_name=region).get_caller_identity()["Account"]

print(f"Region  : {region}")
print(f"Account : {account_id}")

---
## Step 2 — Deploy the travel agent to AgentCore Runtime

The starter toolkit's `Runtime` class configures the agent, builds the container with CodeBuild, and deploys it.

> ⏱️ First deployment takes ~5–10 minutes.

In [ ]:
runtime = Runtime()

runtime.configure(
    entrypoint="agent_app.py",
    agent_name="s07_travel_agent",
    requirements_file="agent_requirements.txt",
    region=region,
    auto_create_execution_role=True,
    auto_create_ecr=True,
    idle_timeout=120,
)

# The deployed agent reads BEDROCK_MODEL_ID (see agent_app.py)
AGENT_MODEL_ID = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")

launch_result = runtime.launch(auto_update_on_conflict=True, env_vars={"BEDROCK_MODEL_ID": AGENT_MODEL_ID})
print(f"\nDeployment started: {launch_result.agent_arn}")

In [ ]:
def wait_for_ready(runtime, name="Agent", poll_seconds=15):
    """Poll until the agent runtime reaches a terminal status."""
    terminal = {"READY", "CREATE_FAILED", "UPDATE_FAILED", "DELETE_FAILED"}
    while True:
        status = runtime.status().endpoint["status"]
        print(f"  {name} status: {status}")
        if status in terminal:
            return status
        time.sleep(poll_seconds)


status = wait_for_ready(runtime, name="Travel Agent")
assert status == "READY", f"Deployment failed with status: {status}"
print("\n✅ Agent is READY")
print(f"   Agent ID  : {launch_result.agent_id}")
print(f"   Agent ARN : {launch_result.agent_arn}")

---
## Step 3 — Create a custom evaluator

The **travel quality** evaluator scores each response on a 5-point scale and gives **Very Poor (0.0)** to any answer outside the travel domain. Its config lives in `travel_quality_metric.json`:
- `modelConfig` — the judge model (Claude Haiku 4.5)
- `instructions` — the judging prompt; `{context}` and `{assistant_turn}` are filled in from the trace
- `ratingScale` — the labels and scores

In [ ]:
eval_client = Evaluation(region=region)

print("Built-in evaluators available:")
for e in eval_client.list_evaluators().get("evaluators", []):
    print(f"  - {e['evaluatorId']}")

In [ ]:
with open("travel_quality_metric.json") as f:
    eval_config = json.load(f)

# Evaluator names must be unique in the Region, so add a short suffix
custom_evaluator = eval_client.create_evaluator(
    name=f"travel_response_quality_{uuid.uuid4().hex[:6]}",
    level="TRACE",
    description="Evaluates travel assistant response quality and scope adherence",
    config=eval_config,
)

evaluator_id = custom_evaluator["evaluatorId"]
print(f"✅ Custom evaluator created: {evaluator_id}")

---
## Step 4 — Invoke the agent with test prompts

All prompts share one `session_id`, so they form one conversation that we can evaluate as a whole.

In [ ]:
session_id = str(uuid.uuid4())
print(f"Session ID: {session_id}\n")

test_prompts = [
    "What flights are available from New York to London?",   # in scope
    "Can you recommend a mid-range hotel in Paris?",         # in scope
    "What is the weather like in London right now?",         # in scope
    "Can you help me write a Python script to sort a list?", # out of scope
]

responses = []
for prompt in test_prompts:
    print(f"📤 Prompt: {prompt}")
    response = runtime.invoke(payload={"prompt": prompt}, session_id=session_id)
    print(f"🤖 Response: {response.get('response', response)}\n")
    responses.append({"prompt": prompt, "response": response})

print(f"✅ Invoked {len(test_prompts)} prompts in session {session_id}")

---
## Step 5 — Wait for traces to be ingested

AgentCore Runtime sends OpenTelemetry traces to CloudWatch. Give them ~3 minutes to arrive before evaluating.

In [ ]:
TRACE_INGESTION_WAIT = 180  # seconds

print(f"Waiting {TRACE_INGESTION_WAIT}s for traces to be ingested...")
for remaining in range(TRACE_INGESTION_WAIT, 0, -30):
    print(f"  {remaining}s remaining...")
    time.sleep(30)
print("✅ Ready to evaluate")

---
## Step 6 — Run evaluations

| Pass | Evaluator | Level | Measures |
|------|-----------|-------|----------|
| A | `Builtin.GoalSuccessRate` | Session | Did the user achieve their goals? |
| B | `Builtin.Correctness` + `Builtin.Helpfulness` | Trace | Per-turn accuracy and usefulness |
| C | custom travel quality | Trace | Scope adherence + quality |

In [ ]:
def show(results, title):
    print(f"=== {title} ===")
    for r in results.results:
        if r.error:
            print(f"⚠️  {r.evaluator_name}: {r.error}")
            continue
        value = r.value if r.value is not None else float("nan")
        emoji = "✅" if value >= 0.75 else ("⚠️" if value >= 0.5 else "❌")
        display(Markdown(f"{emoji} **{r.evaluator_name}** — {r.label} ({value:.2f})\n\n{r.explanation}"))

### Pass A — Session level: Goal Success Rate

In [ ]:
goal_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.GoalSuccessRate"],
)
show(goal_results, "Goal Success Rate (session)")

### Pass B — Trace level: Correctness and Helpfulness

In [ ]:
quality_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.Correctness", "Builtin.Helpfulness"],
)
show(quality_results, "Correctness & Helpfulness (per trace)")

### Pass C — Custom evaluator: travel quality and scope

The out-of-scope Python question should score **Very Poor (0.0)** here, even if the built-in evaluators rated the answer as fine.

In [ ]:
custom_results = eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=[evaluator_id],
)
show(custom_results, "Travel quality — custom (per trace)")

---
## Step 7 — Summary table

In [ ]:
all_results = list(goal_results.results) + list(quality_results.results) + list(custom_results.results)

print(f"{'Evaluator':<40} {'Label':<15} {'Score':>6}")
print("-" * 70)
scores = []
for r in all_results:
    if r.value is None:
        print(f"{r.evaluator_name:<40} {'ERROR':<15} {'-':>6}")
        continue
    scores.append(r.value)
    print(f"{r.evaluator_name:<40} {r.label or '':<15} {r.value:>6.2f}  {'█' * int(r.value * 10)}")

if scores:
    print("-" * 70)
    print(f"{'Average score':<40} {'':<15} {sum(scores) / len(scores):>6.2f}")

---
## Step 8 — Save results to a file

In [ ]:
os.makedirs("eval_output", exist_ok=True)

eval_client.run(
    agent_id=launch_result.agent_id,
    session_id=session_id,
    evaluators=["Builtin.Correctness", evaluator_id],
    output="eval_output/travel_agent_eval_results.json",
)
print("✅ Results saved to eval_output/travel_agent_eval_results.json")

---
## Step 9 — Clean up

In [ ]:
eval_client.delete_evaluator(evaluator_id)
print("Deleted evaluator", evaluator_id)

result = runtime.destroy(delete_ecr_repo=True)
print("Removed:", *result.resources_removed, sep="\n  - ")

---
## What you just did

1. **Deployed** a Strands agent to AgentCore Runtime (no Docker on your machine)
2. **Created** a custom LLM-as-judge evaluator
3. **Invoked** the agent with in-scope and out-of-scope prompts
4. **Evaluated** the session with built-in and custom evaluators
5. **Verified** the custom evaluator catches the out-of-scope answer that generic metrics miss